# DCI-VTON Inference Pipeline
SegFormer + DensePose + AFWM + DCI Diffusion

In [ ]:
import subprocess, sys
import torch as _t
from pathlib import Path

# GPU check
if not _t.cuda.is_available():
    raise RuntimeError('No CUDA GPU available')

_cap = _t.cuda.get_device_capability(0)
_gpu = _t.cuda.get_device_name(0)
print(f'GPU: {_gpu}, capability: sm_{_cap[0]}{_cap[1]}')
print(f'PyTorch: {_t.__version__}')

# PyTorch 2.5+ dropped sm_60 (P100) support entirely — even basic .cuda() fails.
# Detect early and exit fast so worker can retry and get a T4.
if _cap[0] < 7:
    msg = f'GPU_INCOMPATIBLE: {_gpu} sm_{_cap[0]}{_cap[1]} not supported by PyTorch {_t.__version__}. Need sm_70+ (T4/V100/A100).'
    print(msg)
    Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
    Path('/kaggle/working/results/gpu_incompatible.txt').write_text(msg)
    raise RuntimeError(msg)

# Quick CUDA sanity test
_x = _t.ones(2, 2).cuda()
assert _x.sum().item() == 4.0, 'CUDA basic test failed'
print(f'CUDA OK: {_gpu} sm_{_cap[0]}{_cap[1]}')

# Core pip deps — pin transformers to 4.46.3 (>=4.47 breaks smart_apply in CLIPVisionModel)
subprocess.run(['pip', 'install', '-q',
    'omegaconf==2.1.1', 'einops', 'transformers==4.46.3',
    'scikit-image', 'albumentations', 'ftfy', 'regex',
    'fvcore', 'iopath', 'av',
], check=True)
subprocess.run(['pip', 'install', '-q', '--no-deps',
    'open-clip-torch', 'kornia', 'torchmetrics',
], check=True)
# OpenAI CLIP — required by ldm/modules/encoders/modules.py (import clip)
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/openai/CLIP.git',
], check=True)

# pytorch_lightning shim
import types
try:
    import pytorch_lightning
    import pytorch_lightning.utilities as _plu
    if not hasattr(_plu, 'distributed'):
        try:
            from lightning_fabric.utilities.rank_zero import rank_zero_only as _rzo
        except ImportError:
            from pytorch_lightning.utilities.rank_zero import rank_zero_only as _rzo
        _dm = types.ModuleType('pytorch_lightning.utilities.distributed')
        _dm.rank_zero_only = _rzo
        sys.modules['pytorch_lightning.utilities.distributed'] = _dm
        _plu.distributed = _dm
    if not hasattr(_plu, 'seed'):
        _sm = types.ModuleType('pytorch_lightning.utilities.seed')
        _sm.seed_everything = lambda seed=42, **kw: None
        sys.modules['pytorch_lightning.utilities.seed'] = _sm
        _plu.seed = _sm
    print(f'pytorch_lightning: {pytorch_lightning.__version__}')
except Exception as e:
    print(f'PL import warning: {e}')

import torch
print(f'Ready: torch={torch.__version__}, GPU={torch.cuda.get_device_name(0)}')
Path('/kaggle/working/chk_1_done.txt').write_text('done')


In [ ]:
import os, sys, json, shutil, io, collections, traceback
from pathlib import Path
import numpy as np
import torch
import cv2
from PIL import Image

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
Path('/kaggle/working/chk_2_done.txt').write_text('done')


In [ ]:
INPUT_DIR  = Path('/kaggle/input/vton-job-input')
OUTPUT_DIR = Path('/kaggle/working/results')
DCI_DIR    = Path('/kaggle/input/dci-vton-weights')
WORK_DIR   = Path('/kaggle/working/workspace')
SIZE       = 512

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

meta_file = INPUT_DIR / 'job_meta.json'
if not meta_file.exists():
    raise FileNotFoundError(f'job_meta.json missing from {INPUT_DIR}')

meta         = json.loads(meta_file.read_text())
JOB_ID       = meta['job_id']
PERSON_PATH  = str(INPUT_DIR / meta['person_filename'])
GARMENT_PATH = str(INPUT_DIR / meta['garment_filename'])

person_pil  = Image.open(PERSON_PATH).convert('RGB').resize((SIZE, SIZE))
garment_pil = Image.open(GARMENT_PATH).convert('RGB').resize((SIZE, SIZE))
person_np   = np.array(person_pil)
garment_np  = np.array(garment_pil)

print(f'Job ID:  {JOB_ID}')
print(f'Person:  {PERSON_PATH}')
print(f'Garment: {GARMENT_PATH}')
print(f'person_np: {person_np.shape}, garment_np: {garment_np.shape}')
Path('/kaggle/working/chk_3_done.txt').write_text('done')


In [ ]:
# Clone repos
if not Path('/kaggle/working/DCI-VTON-Virtual-Try-On').exists():
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/bcmi/DCI-VTON-Virtual-Try-On.git',
        '/kaggle/working/DCI-VTON-Virtual-Try-On'], check=True)
sys.path.insert(0, '/kaggle/working/DCI-VTON-Virtual-Try-On')

if not Path('/kaggle/working/taming-transformers').exists():
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/CompVis/taming-transformers.git',
        '/kaggle/working/taming-transformers'], check=True)
sys.path.insert(0, '/kaggle/working/taming-transformers')

if not Path('/kaggle/working/PF-AFN').exists():
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/geyuying/PF-AFN.git',
        '/kaggle/working/PF-AFN'], check=True)

import importlib.util
print(f'taming importable: {importlib.util.find_spec("taming") is not None}')
print('Repos ready')
Path('/kaggle/working/chk_4_done.txt').write_text('done')


In [ ]:
# SegFormer segmentation — real body mask
import torch
import torch.nn.functional as F
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
seg_model = SegformerForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes')
seg_model.eval()

def segment(pil_img):
    inputs = processor(images=pil_img, return_tensors='pt')
    with torch.no_grad():
        logits = seg_model(**inputs).logits
    up = F.interpolate(logits, size=(SIZE, SIZE), mode='bilinear', align_corners=False)
    return up.argmax(dim=1).squeeze().numpy()

pred = segment(person_pil)

# ── Garment segment first — needed for neckline detection ────────────────────
g_pred           = segment(garment_pil)
garment_clean_np = garment_np.copy()
garment_clean_np[g_pred == 0] = [128, 128, 128]
garment_clean_pil = Image.fromarray(garment_clean_np)

# ── Person shirt mask (arms excluded — keep arm skin visible) ─────────────────
upper_labels    = [4, 5, 7]
shirt_base_mask = np.isin(pred, upper_labels).astype(np.uint8)

# ── Problem 2: Garment neckline detection ─────────────────────────────────────
garment_fg    = (g_pred != 0)
garment_fg_ys = np.where(garment_fg.any(axis=1))[0]

if len(garment_fg_ys) > 0:
    g_neckline_y      = int(garment_fg_ys.min())
    neckline_row_cols = np.where(garment_fg[g_neckline_y])[0]
    g_neck_width      = int(neckline_row_cols.max() - neckline_row_cols.min()) if len(neckline_row_cols) > 0 else 120
    g_neck_center_x   = int(neckline_row_cols.mean()) if len(neckline_row_cols) > 0 else SIZE // 2
else:
    g_neckline_y, g_neck_width, g_neck_center_x = 100, 120, SIZE // 2

print(f'Garment neckline: y={g_neckline_y}px  width={g_neck_width}px  center_x={g_neck_center_x}')

# ── Collar strip on person ────────────────────────────────────────────────────
shirt_ys = np.where(shirt_base_mask.any(axis=1))[0]

if len(shirt_ys) > 0:
    shirt_top_y    = int(shirt_ys.min())
    shirt_cols     = np.where(shirt_base_mask.any(axis=0))[0]
    person_center_x = int(shirt_cols.mean()) if len(shirt_cols) > 0 else SIZE // 2

    half_w   = max(80, g_neck_width // 2 + 30)
    strip_x1 = max(0,    person_center_x - half_w)
    strip_x2 = min(SIZE, person_center_x + half_w)

    collar_erase_up = max(50, int(g_neckline_y * 0.5))

    collar_strip = np.zeros((SIZE, SIZE), dtype=np.uint8)
    collar_strip[max(0, shirt_top_y - collar_erase_up):min(SIZE, shirt_top_y + 30),
                 strip_x1:strip_x2] = 1
    print(f'Collar strip: erase_up={collar_erase_up}px  x={strip_x1}-{strip_x2}')
else:
    collar_strip = np.zeros((SIZE, SIZE), dtype=np.uint8)
    print('No shirt on person — collar strip skipped')

# ── Final agnostic mask = shirt body (15px dilation) UNION collar strip ───────
k             = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
shirt_dilated = cv2.dilate(shirt_base_mask, k, iterations=1)
agnostic_mask = np.clip(shirt_dilated.astype(np.int32) + collar_strip.astype(np.int32),
                        0, 1).astype(np.uint8)

# blend_mask tight — only for LAB color correction in cell 10
blend_mask = shirt_base_mask.copy()

agnostic_np  = person_np.copy()
agnostic_np[agnostic_mask > 0] = [128, 128, 128]
agnostic_pil = Image.fromarray(agnostic_np)

print(f'Labels found:  {np.unique(pred)}')
print(f'Shirt base:    {shirt_base_mask.sum()}px')
print(f'Agnostic mask: {agnostic_mask.sum()}px')
agnostic_pil.save('/kaggle/working/debug_agnostic.png')
print('Debug: debug_agnostic.png saved')
Path('/kaggle/working/chk_5_done.txt').write_text('done')

In [ ]:
# Real DensePose IUV extraction
import traceback as _tb, os as _os

try:
    import torch as _torch
    _cap = _torch.cuda.get_device_capability(0)
    _gpu_name = _torch.cuda.get_device_name(0)
    _arch_major = _cap[0]
    _arch = f"{_cap[0]}.{_cap[1]}"
    print(f'GPU: {_gpu_name} sm_{_cap[0]}{_cap[1]}')

    # PyTorch 2.x compiled CUDA extensions do NOT work on P100 (sm_60).
    # detectron2 custom ops (ROIAlign, NMS) fail with cudaErrorNoKernelImageForDevice.
    # Fix: use CPU for DensePose when on P100/older GPU.
    _dp_device = 'cpu' if _arch_major < 7 else 'cuda'
    print(f'DensePose device: {_dp_device} (GPU sm_{_cap[0]}{_cap[1]}, need >=7.0 for CUDA ops)')

    _env = _os.environ.copy()
    _env['MAX_JOBS'] = '4'
    if _arch_major >= 7:
        _env['TORCH_CUDA_ARCH_LIST'] = _arch
        _env['FORCE_CUDA'] = '1'

    # Step 1: Install detectron2
    try:
        import detectron2 as _d2_check
        print(f'detectron2 {_d2_check.__version__} already present')
    except ImportError:
        print(f'Installing detectron2 (3-5 min)...')
        _r = subprocess.run(
            ['pip', 'install', '-q',
             'git+https://github.com/facebookresearch/detectron2.git'],
            env=_env, capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f'detectron2 install failed\nSTDOUT:{_r.stdout[-500:]}\nSTDERR:{_r.stderr[-500:]}')
        print('detectron2 installed!')

    # Step 2: Clone repo for DensePose config + source
    if not Path('/kaggle/working/detectron2_repo').exists():
        subprocess.run(['git', 'clone', '--depth=1',
            'https://github.com/facebookresearch/detectron2.git',
            '/kaggle/working/detectron2_repo'], check=True)
    print('Step2: repo ready')

    # Step 3: DensePose projects path
    sys.path.insert(0, '/kaggle/working/detectron2_repo/projects/DensePose')

    # Step 4: Fresh import
    for _k in list(sys.modules.keys()):
        if 'detectron2' in _k or 'densepose' in _k:
            del sys.modules[_k]

    import detectron2
    print(f'Step4: detectron2 {detectron2.__version__}')
    import densepose
    print('Step4: densepose OK')

    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor
    from detectron2.data import DatasetCatalog, MetadataCatalog
    from densepose import add_densepose_config

    for k in [k for k in DatasetCatalog if 'densepose' in k]:
        DatasetCatalog.remove(k)
        try: MetadataCatalog.remove(k)
        except: pass

    # Step 5: Download weights
    dp_weights = Path('/kaggle/working/densepose_rcnn_R_50_FPN_s1x.pkl')
    if not dp_weights.exists():
        import urllib.request
        url = 'https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl'
        print('Downloading DensePose weights (~250MB)...')
        urllib.request.urlretrieve(url, str(dp_weights))
        print(f'Downloaded: {dp_weights.stat().st_size // 1024 // 1024} MB')
    else:
        print(f'Step5: weights exist {dp_weights.stat().st_size // 1024 // 1024} MB')

    # Step 6: Setup predictor — use CPU on P100 to avoid CUDA compiled ops issue
    cfg = get_cfg()
    add_densepose_config(cfg)
    cfg.merge_from_file('/kaggle/working/detectron2_repo/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml')
    cfg.MODEL.WEIGHTS = str(dp_weights)
    cfg.MODEL.DEVICE  = _dp_device   # 'cpu' for P100, 'cuda' for T4/A100
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
    cfg.freeze()
    print(f'Step6: creating DefaultPredictor on {_dp_device}...')
    dp_predictor = DefaultPredictor(cfg)
    print(f'Step6: DefaultPredictor OK on {_dp_device}')

    def get_densepose_iuv(predictor, person_np_img, size=512):
        outputs   = predictor(person_np_img[:, :, ::-1])
        instances = outputs['instances']
        iuv_full  = np.zeros((size, size, 3), dtype=np.float32)
        if len(instances) == 0:
            print('Warning: no person detected, using zero IUV')
            return iuv_full
        best       = instances.scores.cpu().numpy().argmax()
        result     = instances.pred_densepose[best]
        bbox       = instances.pred_boxes.tensor.cpu().numpy()[best]
        fine_segm  = result.fine_segm.cpu()
        u_map      = result.u.cpu()
        v_map      = result.v.cpu()
        part_idx_t = fine_segm.argmax(dim=0)
        idx_exp    = part_idx_t.unsqueeze(0)
        u_vals     = u_map.gather(0, idx_exp).squeeze(0).numpy()
        v_vals     = v_map.gather(0, idx_exp).squeeze(0).numpy()
        part_idx   = part_idx_t.numpy().astype(np.float32)
        x1,y1,x2,y2 = map(int, bbox)
        bh = max(1, y2-y1); bw = max(1, x2-x1)
        I_r = cv2.resize(part_idx, (bw, bh))
        U_r = cv2.resize(u_vals,   (bw, bh))
        V_r = cv2.resize(v_vals,   (bw, bh))
        I_f = np.zeros((size, size), dtype=np.float32)
        U_f = np.zeros((size, size), dtype=np.float32)
        V_f = np.zeros((size, size), dtype=np.float32)
        y1c=max(0,y1); y2c=min(size,y2)
        x1c=max(0,x1); x2c=min(size,x2)
        dh=y2c-y1c; dw=x2c-x1c
        I_f[y1c:y2c, x1c:x2c] = I_r[:dh,:dw] / 112.0
        U_f[y1c:y2c, x1c:x2c] = U_r[:dh,:dw]
        V_f[y1c:y2c, x1c:x2c] = V_r[:dh,:dw]
        return np.stack([I_f, U_f, V_f], axis=-1)

    print('Step7: running DensePose inference...')
    densepose_iuv = get_densepose_iuv(dp_predictor, person_np)
    print(f'IUV shape: {densepose_iuv.shape}')
    print(f'I range: {densepose_iuv[:,:,0].min():.3f} -> {densepose_iuv[:,:,0].max():.3f}')
    print(f'U range: {densepose_iuv[:,:,1].min():.3f} -> {densepose_iuv[:,:,1].max():.3f}')
    Path('/kaggle/working/chk_6_done.txt').write_text('done')
    print('chk_6 DONE')

except Exception as _exc:
    _err_txt = _tb.format_exc()
    print('=== CHK_6 FAILED ===')
    print(_err_txt[-3000:])
    Path('/kaggle/working/chk_6_error.txt').write_text(_err_txt)
    raise


In [ ]:
# AFWM warp — garment deformed to body shape
import torch
import torch.nn.functional as F
import torchvision.transforms as T

corr_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F

def FunctionCorrelation(tenFirst, tenSecond, intStride=1):
    B, C, H, W = tenFirst.shape
    D = 3
    tenSecond_pad = F.pad(tenSecond, [D, D, D, D])
    result = []
    for dy in range(2*D+1):
        for dx in range(2*D+1):
            shifted = tenSecond_pad[:, :, dy:dy+H, dx:dx+W]
            corr = (tenFirst * shifted).mean(dim=1, keepdim=True)
            result.append(corr)
    return torch.cat(result, dim=1)

class Correlation(nn.Module):
    def __init__(self, max_displacement=4, *args, **kwargs):
        super().__init__()
        self.max_displacement = max_displacement
    def forward(self, input1, input2):
        return FunctionCorrelation(input1, input2)
'''

if not Path('/kaggle/working/PF-AFN').exists():
    subprocess.run(['git', 'clone', '--depth=1',
        'https://github.com/geyuying/PF-AFN.git',
        '/kaggle/working/PF-AFN'], check=True)

Path('/kaggle/working/PF-AFN/PF-AFN_test/models/__init__.py').touch()
Path('/kaggle/working/PF-AFN/PF-AFN_test/models/correlation').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/PF-AFN/PF-AFN_test/models/correlation/__init__.py').touch()
Path('/kaggle/working/PF-AFN/PF-AFN_test/models/correlation/correlation.py').write_text(corr_code)

for key in list(sys.modules.keys()):
    if any(x in key for x in ['afwm', 'correlation', 'networks']):
        del sys.modules[key]

sys.path.insert(0, '/kaggle/working/PF-AFN/PF-AFN_test')
from models.afwm import AFWM
from models.networks import load_checkpoint

class WarpOpt:
    label_nc  = 13
    gpu_ids   = [0]
    batchSize = 1
    fineSize  = 512
    isTrain   = False

warp_opt   = WarpOpt()
warp_model = AFWM(warp_opt, 3 + warp_opt.label_nc)

warp_pth = DCI_DIR / 'warp_viton.pth'
if not warp_pth.exists():
    dl_env = os.environ.copy()
    subprocess.run(
        ['kaggle', 'datasets', 'download', 'aisubscrip2/dci-vton-weights',
         '--file', 'warp_viton.pth', '-p', '/kaggle/working/weights', '--force'],
        env=dl_env, capture_output=True, text=True, timeout=300
    )
    import zipfile
    for z in Path('/kaggle/working/weights').glob('*.zip'):
        with zipfile.ZipFile(z) as zf: zf.extractall('/kaggle/working/weights')
        z.unlink()
    warp_pth = Path('/kaggle/working/weights/warp_viton.pth')

load_checkpoint(warp_model, str(warp_pth))
warp_model.eval().cuda()
print('AFWM loaded!')

# Build 13-channel parse map
seg_to_parse = {0:0, 2:1, 11:2, 3:3, 4:3, 7:3, 5:4, 6:5,
                8:6, 14:7, 15:8, 12:9, 13:10, 9:11, 10:12}
parse_map = np.zeros((13, SIZE, SIZE), dtype=np.float32)
for seg_lbl, parse_ch in seg_to_parse.items():
    parse_map[parse_ch][pred == seg_lbl] = 1.0
parse_map[3][agnostic_mask > 0] = 0
parse_map[7][agnostic_mask > 0] = 0
parse_map[8][agnostic_mask > 0] = 0
parse_map[2][agnostic_mask > 0] = 1.0

warp_tf = T.Compose([T.ToTensor(), T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])

# PF-AFN AFWM expects: [agnostic_image(3ch), parse_map(13ch)] = 16ch
# Previously we passed [parse(13), densepose(3)] — wrong, caused bad warping.
agnostic_t = warp_tf(agnostic_pil).unsqueeze(0).cuda()   # 3ch agnostic person
parse_t    = torch.from_numpy(parse_map).unsqueeze(0).cuda()
cond_input = torch.cat([agnostic_t, parse_t], dim=1)      # (1,16,512,512)
garment_t  = warp_tf(garment_clean_pil).unsqueeze(0).cuda()

with torch.no_grad():
    warped_cloth, _ = warp_model(cond_input, garment_t)
    warped_np = warped_cloth.squeeze().permute(1,2,0).cpu().numpy()
    warped_np = ((warped_np + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
    warped_garment_pil = Image.fromarray(warped_np)

warped_garment_pil.save('/kaggle/working/warped_garment.png')
print('AFWM warp done!')
Path('/kaggle/working/chk_7_done.txt').write_text('done')


In [ ]:
# Load DCI-VTON model — CLIP patch + VGG fix
import os, collections
import torchvision.models as tv_models
from omegaconf import OmegaConf

# Pin transformers to last known-good version for CLIPVisionModel eager-attn support.
# transformers>=4.47 broke PreTrainedModel.initialize_weights (smart_apply signature).
subprocess.run(['pip', 'install', '-q', 'transformers==4.46.3'], check=True)
for key in list(sys.modules.keys()):
    if 'transformers' in key:
        del sys.modules[key]

# Patch CLIPVisionModel to use eager attention (avoids flash-attn KeyError)
from transformers import CLIPVisionModel as _CLIP
_orig_fp = _CLIP.from_pretrained.__func__
@classmethod
def _patched_fp(cls, *args, **kwargs):
    kwargs.setdefault('attn_implementation', 'eager')
    return _orig_fp(cls, *args, **kwargs)
_CLIP.from_pretrained = _patched_fp

import ldm.modules.encoders.modules as _enc_mod
_enc_mod.CLIPVisionModel = _CLIP

# Build VGG weights with DCI key names (conv1_1, conv1_2, ...)
os.makedirs('/kaggle/working/models/vgg', exist_ok=True)
vgg_pth = '/kaggle/working/models/vgg/vgg19_conv.pth'
if not Path(vgg_pth).exists():
    idx_to_name = {
        0:'conv1_1', 2:'conv1_2',
        5:'conv2_1', 7:'conv2_2',
        10:'conv3_1', 12:'conv3_2', 14:'conv3_3', 16:'conv3_4',
        19:'conv4_1', 21:'conv4_2', 23:'conv4_3', 25:'conv4_4',
        28:'conv5_1', 30:'conv5_2', 32:'conv5_3', 34:'conv5_4',
    }
    vgg19 = tv_models.vgg19(weights=tv_models.VGG19_Weights.DEFAULT)
    sd = collections.OrderedDict()
    for idx, name in idx_to_name.items():
        layer = vgg19.features[idx]
        sd[f'{name}.weight'] = layer.weight.data.clone()
        sd[f'{name}.bias']   = layer.bias.data.clone()
    torch.save(sd, vgg_pth)
    print(f'VGG weights saved: {len(sd)} tensors')

# Set CWD so relative path 'models/vgg/vgg19_conv.pth' resolves
os.chdir('/kaggle/working')

from ldm.util import instantiate_from_config
from ldm.models.diffusion.ddim import DDIMSampler

CONFIG_PATH = '/kaggle/working/DCI-VTON-Virtual-Try-On/configs/viton512.yaml'
CKPT_PATH   = DCI_DIR / 'viton512.ckpt'

# Download viton512.ckpt if not in dataset mount
if not CKPT_PATH.exists():
    wt_dir = Path('/kaggle/working/weights')
    wt_dir.mkdir(exist_ok=True)
    dl_env = os.environ.copy()
    r = subprocess.run(
        ['kaggle','datasets','download','aisubscrip2/dci-vton-weights',
         '--file','viton512.ckpt','-p',str(wt_dir),'--force'],
        env=dl_env, capture_output=True, text=True, timeout=600
    )
    import zipfile
    for z in wt_dir.glob('*.zip'):
        with zipfile.ZipFile(z) as zf: zf.extractall(str(wt_dir))
        z.unlink()
    CKPT_PATH = wt_dir / 'viton512.ckpt'

print(f'Loading checkpoint: {CKPT_PATH}')
config = OmegaConf.load(CONFIG_PATH)
pl_sd  = torch.load(str(CKPT_PATH), map_location='cpu', weights_only=False)
model  = instantiate_from_config(config.model)
missing, unexpected = model.load_state_dict(pl_sd['state_dict'], strict=False)
print(f'Missing: {len(missing)}  Unexpected: {len(unexpected)}')
model  = model.cuda().eval()
sampler = DDIMSampler(model)
print('DCI-VTON model loaded!')
Path('/kaggle/working/chk_8_done.txt').write_text('done')


In [ ]:
# DCI-VTON inference
import hashlib
import torchvision
from torchvision.transforms import Resize

device = torch.device('cuda')
H, W, C, f = 512, 512, 4, 8

# Problem 1 fix: job_id based seed — same job = same result, different jobs = different result
_seed = int(hashlib.md5(JOB_ID.encode()).hexdigest()[:8], 16) % (2**31)
torch.manual_seed(_seed)
np.random.seed(_seed)
print(f'Seed: {_seed}  (job_id: {JOB_ID})')

def to_tensor(img):
    t = torchvision.transforms.ToTensor()(img)
    return torchvision.transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])(t).unsqueeze(0)

def to_clip(img):
    img = img.resize((224,224), Image.LANCZOS)
    t = torchvision.transforms.ToTensor()(img)
    return torchvision.transforms.Normalize(
        (0.48145466,0.4578275,0.40821073),(0.26862954,0.26130258,0.27577711)
    )(t).unsqueeze(0)

inpaint_image_t = to_tensor(agnostic_pil).to(device)
feat_tensor     = to_tensor(warped_garment_pil).to(device)
ref_tensor      = to_clip(garment_clean_pil).to(device)

mask_np_float       = agnostic_mask.astype(np.float32)
mask_t              = torch.from_numpy(mask_np_float).unsqueeze(0).unsqueeze(0).to(device)
inpaint_mask_tensor = 1.0 - mask_t  # DCI: 0=generate, 1=keep

model.eval()

with torch.no_grad():
    c  = model.get_learned_conditioning(ref_tensor.to(torch.float16))
    c  = model.proj_out(c)
    uc = model.learnable_vector.repeat(ref_tensor.size(0), 1, 1)

    z_inpaint = model.encode_first_stage(inpaint_image_t)
    z_inpaint = model.get_first_stage_encoding(z_inpaint).detach()

    warp_feat = model.encode_first_stage(feat_tensor)
    warp_feat = model.get_first_stage_encoding(warp_feat).detach()

    mask_latent = Resize([z_inpaint.shape[-2], z_inpaint.shape[-1]])(inpaint_mask_tensor)

    test_model_kwargs = {
        'inpaint_image': z_inpaint,
        'inpaint_mask':  mask_latent,
    }

    # N_STEPS=20: lower noise → garment texture/color preserved, no hallucination
    N_STEPS = 20
    sampler.make_schedule(ddim_num_steps=50, ddim_eta=0.0, verbose=False)

    total      = sampler.ddim_timesteps.shape[0]
    subset_end = int(min(N_STEPS / total, 1) * total) - 1
    T_START    = int(sampler.ddim_timesteps[subset_end - 1])
    print(f'total={total}, subset_end={subset_end}, T_START={T_START}')

    ts         = torch.full((1,), T_START, device=device, dtype=torch.long)
    start_code = model.q_sample(warp_feat, ts)
    print(f'x_T noise: t={T_START}, α̅={model.alphas_cumprod[T_START].item():.4f}')

    samples, _ = sampler.ddim_sampling(
        cond=c,
        shape=(1, C, H//f, W//f),
        x_T=start_code,
        timesteps=N_STEPS,
        unconditional_guidance_scale=12.0,
        unconditional_conditioning=uc,
        test_model_kwargs=test_model_kwargs,
    )

    x_out = model.decode_first_stage(samples)
    x_out = torch.clamp((x_out + 1.0) / 2.0, 0.0, 1.0)
    x_out = x_out.cpu().permute(0,2,3,1).numpy()[0]

print('DCI inference complete!')
Path('/kaggle/working/chk_9_done.txt').write_text('done')

In [ ]:
# Save + export final result
import cv2 as _cv2
from skimage.exposure import match_histograms

result_np = (x_out * 255).astype(np.uint8)
Image.fromarray(result_np).save('/kaggle/working/dci_raw_output.png')
print('Debug: dci_raw_output.png saved')

shirt_mask = blend_mask > 0
garment_fg = g_pred != 0

if shirt_mask.sum() > 200 and garment_fg.sum() > 200:
    result_lab  = _cv2.cvtColor(result_np,             _cv2.COLOR_RGB2LAB).astype(np.float32)
    garment_lab = _cv2.cvtColor(np.array(garment_pil), _cv2.COLOR_RGB2LAB).astype(np.float32)

    corrected_lab = result_lab.copy()

    # L channel: 35% partial shift — enough for correct darkness, not enough for product shadow
    ref_L = garment_lab[:, :, 0][garment_fg].mean()
    res_L = result_lab[:, :, 0][shirt_mask].mean()
    dL = (ref_L - res_L) * 0.35
    corrected_lab[:, :, 0][shirt_mask] = np.clip(result_lab[:, :, 0][shirt_mask] + dL, 0, 255)
    print(f'L shift 35%: garment={ref_L:.1f} result={res_L:.1f} dL={dL:.1f}')

    # a+b channels: histogram match (exact hue)
    res_a_pixels = result_lab[:, :, 1][shirt_mask]
    res_b_pixels = result_lab[:, :, 2][shirt_mask]
    ref_a_pixels = garment_lab[:, :, 1][garment_fg]
    ref_b_pixels = garment_lab[:, :, 2][garment_fg]

    corrected_lab[:, :, 1][shirt_mask] = match_histograms(res_a_pixels, ref_a_pixels)
    corrected_lab[:, :, 2][shirt_mask] = match_histograms(res_b_pixels, ref_b_pixels)

    print(f'Garment LAB mean: a={ref_a_pixels.mean():.1f} b={ref_b_pixels.mean():.1f}')
    print(f'Result  LAB mean: a={res_a_pixels.mean():.1f} b={res_b_pixels.mean():.1f}')

    color_corrected_np = _cv2.cvtColor(corrected_lab.astype(np.uint8), _cv2.COLOR_LAB2RGB)
else:
    print('Color correction skipped')
    color_corrected_np = result_np

# Composite: shirt_base_mask only (no collar strip) — neck area keeps original person skin
k_comp = _cv2.getStructuringElement(_cv2.MORPH_ELLIPSE, (7, 7))
composite_mask = _cv2.dilate(shirt_base_mask, k_comp, iterations=1).astype(np.float32)
composite_mask = _cv2.GaussianBlur(composite_mask, (11, 11), 3.0)
composite_mask = composite_mask[:, :, np.newaxis]

final_np = (color_corrected_np.astype(np.float32) * composite_mask +
            person_np.astype(np.float32) * (1.0 - composite_mask)).astype(np.uint8)
print('Composite: DCI shirt + original person (shirt_base_mask, soft edge)')

final_img = Image.fromarray(final_np)
final_img.save('/kaggle/working/final_output.png')
print('Debug: final_output.png saved')

output_file = OUTPUT_DIR / f'{JOB_ID}.jpg'
final_img.save(str(output_file), 'JPEG', quality=95)
print(f'Result saved: {output_file}')

root_result = Path(f'/kaggle/working/!{JOB_ID}.jpg')
import shutil
shutil.copy(str(output_file), str(root_result))
print(f'Root copy: {root_result}')

(OUTPUT_DIR / 'result_meta.json').write_text(json.dumps({
    'job_id': JOB_ID, 'status': 'completed', 'output_file': f'{JOB_ID}.jpg'
}))
print('Pipeline complete!')
Path('/kaggle/working/chk_10_done.txt').write_text('done')